**Pixeltable replaces 5-8 services with one table where every modality is a first-class column type.** Extensible via `@pxt.udf`, `@pxt.uda`, and `@pxt.query`.

| Instead of | Pixeltable |
|---|---|
| PostgreSQL / MySQL | `pxt.create_table()`, versioned automatically |
| Pinecone / Weaviate | `add_embedding_index()`, always in sync |
| S3 / boto3 | `pxt.Image`, `Video`, `Audio`, `Document` types |
| Airflow / Prefect | Computed columns trigger on insert |
| LangChain / LlamaIndex | `@pxt.query` + `.similarity()` |

[20+ AI providers](https://docs.pixeltable.com/integrations/frameworks) built in.
[llms.txt](https://docs.pixeltable.com/llms.txt) |
[MCP Server](https://github.com/pixeltable/mcp-server-pixeltable-developer) |
[Starter Kit](https://github.com/pixeltable/pixeltable-starter-kit)

```bash
pip install pixeltable google-genai 'fastapi[standard]'
pip install torch transformers  # optional, for object detection
```

In [1]:
import os, getpass

if 'GEMINI_API_KEY' not in os.environ and 'GOOGLE_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API Key: ')

import pixeltable as pxt
from pixeltable.functions import gemini

BASE_URL = 'https://raw.githubusercontent.com/pixeltable/pixeltable/release/docs/resources'

## 1. Store: Multimodal Tables

Video, audio, images, and documents are first-class column types. `pip install pixeltable` is all you need.

In [2]:
from pixeltable.functions.uuid import uuid7

pxt.drop_dir('demo', force=True, if_not_exists='ignore')
pxt.create_dir('demo')

videos = pxt.create_table(
    'demo/videos',
    {'id': uuid7(), 'video': pxt.Video, 'title': pxt.String},
    primary_key='id',
)
videos

Connected to Pixeltable database at: postgresql+psycopg://postgres:@/pixeltable?host=/Users/pjlb/.pixeltable/pgdata
Found an existing Pixeltable dashboard at: http://localhost:22089
Created directory 'demo'.
Created table 'videos'.


table 'demo/videos'

 Column Name    Type  Source Computed With Comment
--------------------------------------------------
       video   Video  videos                      
       title  String  videos

## 2. Orchestrate: AI as Computed Columns

Add a computed column; Pixeltable calls Gemini on every insert, caches results, retries failures, keeps embeddings in sync.

In [3]:
videos.add_computed_column(
    response=gemini.generate_content(
        [videos.video, 'Describe this video in detail.'], model='gemini-3-flash-preview'
    )
)

videos.add_computed_column(
    description=videos.response.candidates[0].content.parts[0].text.astype(pxt.String)
)

videos.add_embedding_index('description', embedding=gemini.embed_content.using(model='gemini-embedding-2-preview'))

Added 0 column values with 0 errors in 0.01 s
Added 0 column values with 0 errors in 0.02 s


## 3. Insert: One Call Triggers the Full Pipeline

`insert()` downloads videos, runs Gemini, extracts text, computes embeddings. Open the **Dashboard** to watch in real time.

In [4]:
videos.insert([
    {'video': f'{BASE_URL}/bangkok.mp4', 'title': 'Bangkok Street Tour'},
    {'video': f'{BASE_URL}/The-Pursuit-of-Happiness-Video-Extract.mp4', 'title': 'The Pursuit of Happiness'},
])

ERROR:asyncio:Fatal error on SSL transport
protocol: <asyncio.sslproto.SSLProtocol object at 0x1562dd240>
transport: <_SelectorSocketTransport closing fd=85>
Traceback (most recent call last):
  File "/opt/miniconda3/envs/pxt/lib/python3.10/asyncio/selector_events.py", line 924, in write
    n = self._sock.send(data)
OSError: [Errno 9] Bad file descriptor

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/miniconda3/envs/pxt/lib/python3.10/asyncio/sslproto.py", line 690, in _process_write_backlog
    self._transport.write(chunk)
  File "/opt/miniconda3/envs/pxt/lib/python3.10/asyncio/selector_events.py", line 930, in write
    self._fatal_error(exc, 'Fatal write error on socket transport')
  File "/opt/miniconda3/envs/pxt/lib/python3.10/asyncio/selector_events.py", line 725, in _fatal_error
    self._force_close(exc)
  File "/opt/miniconda3/envs/pxt/lib/python3.10/asyncio/selector_events.py", line 737, in _force_close
 

Inserted 2 rows with 0 errors in 21.59 s (0.09 rows/s)


2 rows inserted.

In [5]:
videos = pxt.get_table('demo/videos')
videos.select(videos.title, videos.description).collect()

ERROR:asyncio:Fatal error on SSL transport
protocol: <asyncio.sslproto.SSLProtocol object at 0x1562deb90>
transport: <_SelectorSocketTransport closing fd=84>
Traceback (most recent call last):
  File "/opt/miniconda3/envs/pxt/lib/python3.10/asyncio/selector_events.py", line 924, in write
    n = self._sock.send(data)
OSError: [Errno 9] Bad file descriptor

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/miniconda3/envs/pxt/lib/python3.10/asyncio/sslproto.py", line 690, in _process_write_backlog
    self._transport.write(chunk)
  File "/opt/miniconda3/envs/pxt/lib/python3.10/asyncio/selector_events.py", line 930, in write
    self._fatal_error(exc, 'Fatal write error on socket transport')
  File "/opt/miniconda3/envs/pxt/lib/python3.10/asyncio/selector_events.py", line 725, in _fatal_error
    self._force_close(exc)
  File "/opt/miniconda3/envs/pxt/lib/python3.10/asyncio/selector_events.py", line 737, in _force_close
 

title,description
Bangkok Street Tour,"In this high-angle shot, we are looking at a busy multi-lane street in what appears to be a major city in Thailand, given the presence of tuk-tuks and colorful taxis. The road is divided by a narrow median with some small green bushes. Traffic is moving in both directions, though the lanes moving toward the camera are more prominent. On the left side of the street, several large buildings feature massive billboards and advertisements, including one for a beauty product. Pedestrians can be s ...... oxes, weave through the traffic. Vehicles driving toward the camera include silver and white cars, as well as several pink taxis and another green-and-yellow taxi. In the distance, a pedestrian bridge spans across the road, and tall skyscrapers are visible against a slightly hazy sky. On the right side of the road, there are more buildings, some trees, and a series of small pink tents or stalls set up along the sidewalk. The overall atmosphere is that of a typical, bustling urban afternoon."
The Pursuit of Happiness,"The video captures a poignant moment from the movie ""The Pursuit of Happyness,"" featuring Chris Gardner (Will Smith) and Jay Twistle (Brian Howe). The scene takes place in a professional office lobby with wood-paneled walls. Chris Gardner is seen looking disheveled, with white paint splatters on his face and wearing casual, somewhat worn clothes. Jay Twistle, dressed in a formal suit and tie, approaches him with a wide, encouraging smile. Mr. Twistle calls out, ""Chris..."" as he walks towa ...... ingering anxiety. Mr. Twistle’s warmth is evident as he says, ""Hey, now you can call me Jay. We'll talk to you soon,"" before turning to walk toward the elevators. This simple gesture of asking to be called by his first name signifies a major turning point, indicating that Chris has successfully impressed him and likely secured a spot in the competitive internship program despite his unconventional appearance. The scene ends with Jay walking away, leaving a visibly moved Chris in the lobby."


## 4. Retrieve: Semantic Search

Embedding index stays in sync automatically. No separate vector DB.

In [6]:
sim = videos.description.similarity(string='street food')
videos.order_by(sim, asc=False).limit(5).select(videos.title, sim).collect()

title,similarity
The Pursuit of Happiness,0.368
Bangkok Street Tour,0.112


## 5. Experiment on Media Data

Extract a frame, run DETR object detection, draw bounding boxes, all in one expression. Change `timestamp` and re-run to explore different frames.

In [7]:
from pixeltable.functions import huggingface

videos.select(
    videos.title,
    detections=huggingface.detr_for_object_detection(
        videos.video.extract_frame(timestamp=2.0), model_id='facebook/detr-resnet-50',
    ),
).collect()

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


title,detections
Bangkok Street Tour,"{""scores"": [0.589, 0.808, 0.996, 0.641, 0.917, 0.986, ..., 0.979, 0.921, 0.928, 0.898, 0.995, 0.657], ""labels"": [1, 3, 4, 3, 3, 3, ..., 3, 3, 4, 1, 3, 3], ""label_text"": [""person"", ""car"", ""motorcycle"", ""car"", ""car"", ""car"", ..., ""car"", ""car"", ""motorcycle"", ""person"", ""car"", ""car""], ""boxes"": [[39.963, 338.669, 72.671, 362.745], [609.464, 225.755, 637.114, 248.488], [268.929, 616.383, 400.707, 713.272], [675.75, 191.805, 692.807, 212.131], [541.478, 226.507, 579.56, 258.582], [561.88, 257.446, 604.472, 286.533], ..., [635.494, 238.111, 669.913, 265.062], [653.239, 212.862, 675.404, 232.896], [453.433, 543.869, 610.01, 703.072], [265.99, 291.568, 281.883, 326.052], [840.647, 325.342, 901.879, 377.961], [699.855, 189.767, 719.899, 211.171]]}"
The Pursuit of Happiness,"{""scores"": [0.93, 0.804, 0.998, 0.818, 0.514, 0.98, 0.988, 0.994, 0.856, 0.998], ""labels"": [1, 31, 32, 1, 32, 1, 1, 1, 1, 1], ""label_text"": [""person"", ""handbag"", ""tie"", ""person"", ""tie"", ""person"", ""person"", ""person"", ""person"", ""person""], ""boxes"": [[1008.568, 195.299, 1197.513, 615.01], [1120.287, 389.688, 1198.117, 458.87], [887.229, 304.996, 941.166, 513.493], [428.474, 195.906, 519.1, 545.015], [1076.862, 286.451, 1094.088, 337.522], [961.815, 186.835, 1009.2, 284.666], [579.152, 310.525, 708.65, 405.497], [48.105, 93.978, 498.734, 633.37], [1007.286, 195.206, 1196.64, 482.595], [693.094, 115.559, 1165.177, 631.205]]}"


## 6. Serve: Queries Become API Endpoints

`@pxt.query` becomes an HTTP endpoint via `FastAPIRouter`. In production, use `pxt serve service.toml`. See [HTTP Serving](https://docs.pixeltable.com/howto/deployment/serving).

In [8]:
import json
import fastapi
from fastapi.testclient import TestClient
from pixeltable.serving import FastAPIRouter

@pxt.query
def search_videos(query_text: str, limit: int = 5):
    sim = videos.description.similarity(string=query_text)
    return videos.order_by(sim, asc=False).limit(limit).select(videos.title, videos.description, sim)

app = fastapi.FastAPI()
router = FastAPIRouter()
router.add_query_route(path='/search', query=search_videos)
router.add_insert_route(videos, path='/ingest', inputs=['video', 'title'], outputs=['title', 'description'])
app.include_router(router)

client = TestClient(app)

In [9]:
resp = client.post('/search', json={'query_text': 'street food', 'limit': 2})
resp.json()

{'rows': [{'title': 'The Pursuit of Happiness',
   'description': 'The video captures a poignant moment from the movie "The Pursuit of Happyness," featuring Chris Gardner (Will Smith) and Jay Twistle (Brian Howe). \n\nThe scene takes place in a professional office lobby with wood-paneled walls. Chris Gardner is seen looking disheveled, with white paint splatters on his face and wearing casual, somewhat worn clothes. Jay Twistle, dressed in a formal suit and tie, approaches him with a wide, encouraging smile.\n\nMr. Twistle calls out, "Chris..." as he walks toward him. He continues, "I don\'t know how you did it dressed as a garbage man, but you really pulled it off in there." This comment highlights the contrast between Chris\'s appearance and his performance during what was likely a high-stakes meeting or interview. \n\nChris responds with a humble "Thank you very much, Mr. Twistle," his expression a mix of relief and lingering anxiety. \n\nMr. Twistle’s warmth is evident as he says, 

In [10]:
resp = client.post('/ingest', json={'video': f'{BASE_URL}/bangkok.mp4', 'title': 'Test Upload'})
resp.json()

Inserted 1 row with 0 errors in 14.31 s (0.07 rows/s)


{'title': 'Test Upload',
 'description': 'An elevated, wide-angle shot reveals a bustling multi-lane thoroughfare in a densely developed urban area, likely Bangkok, Thailand. The street is filled with a constant flow of traffic, featuring a mix of modern cars, motorcycles, colorful taxis in vibrant shades of pink and yellow-green, and several iconic three-wheeled tuk-tuks. \n\nOn the left side of the frame, several multi-story buildings rise, their facades adorned with large, colorful advertising billboards featuring various products and faces. Storefronts at the street level, such as one labeled "B-I-G C," are visible. On the right, another large commercial building is lined with a row of white and red pointed tents along the sidewalk. A pedestrian overpass spans the wide road in the distance, connecting the two sides of the busy street.\n\nThe sky is bright and clear, casting an even light over the scene. In the far background, the silhouettes of distant skyscrapers and a prominent t

## Bonus: Cloud Storage (Optional)

Free managed bucket with Pixeltable Cloud. Set two config values; computed media flows to cloud. See [Cloud Services](https://docs.pixeltable.com/use-cases/services).

```toml
# ~/.pixeltable/config.toml
[pixeltable]
api_key = "your-pixeltable-cloud-api-key"
output_media_dest = "pxtfs://yourorg:yourdb/home"
```

## Summary

| You write | Pixeltable does |
|---|---|
| `pxt.Image`, `pxt.Video`, `pxt.Document` columns | Stores media, caches from URLs |
| `add_computed_column(fn(...))` | Runs incrementally, caches, retries |
| `add_embedding_index(column)` | Vector storage, keeps index in sync |
| `@pxt.udf` / `@pxt.query` | Reusable functions with dependency tracking |
| `table.insert(...)` | Triggers all dependent computations |
| `table.select(...).collect()` | Returns structured + unstructured data |
| *(automatic)* | Versions all data and schema changes |

| Primitive | Description |
|---|---|
| **Store** | `Image`, `Video`, `Audio`, `Document` column types |
| **Orchestrate** | AI providers as computed columns |
| **Iterate** | Views with iterators (frames, chunks, segments) |
| **Index** | Embedding indexes, always in sync |
| **Extend** | `@pxt.udf`, any Python library |
| **Agents** | `@pxt.tools` + `invoke_tools()` |
| **Query** | `.sample()`, `.select()`, on-the-fly expressions |
| **Serve** | `FastAPIRouter` or `pxt serve` |
| **Version** | `history()`, `revert()`, time-travel |
| **Import/Export** | CSV, JSON, Parquet, PyTorch, COCO, LanceDB, SQL, HuggingFace |

### Links

- [10-Minute Tour](https://docs.pixeltable.com/overview/ten-minute-tour)
- [Starter Kit](https://github.com/pixeltable/pixeltable-starter-kit) (FastAPI + React reference app)
- [Cookbooks](https://docs.pixeltable.com/howto/cookbooks) (50+ recipes)
- [Docs](https://docs.pixeltable.com/)
- [GitHub](https://github.com/pixeltable/pixeltable) (Apache 2.0)